# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

# TASK 1

In [46]:
import open3d as o3d
import numpy as np
import copy

# helper function for drawing
# If you want it to be more clear set recolor=True
def draw_registrations(source, target, transformation = None, recolor = False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if(transformation is not None):
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

Match R1 and R2

In [47]:
source_R = o3d.io.read_point_cloud("global_registration/r1.pcd")
target_R = o3d.io.read_point_cloud("global_registration/r2.pcd")

# Used for downsampling.
voxel_size_R = 0.05

# Show models side by side
draw_registrations(source_R, target_R)

In [48]:
####
# Downsample and find features here
####
# Función auxiliar para preprocesar la nube de puntos (downsampling, normales y FPFH)
def preprocess_point_cloud(pcd, voxel_size):
    # 1. Reducción de muestreo (Voxel downsampling)
    pcd_down = pcd.voxel_down_sample(voxel_size)
    
    # 2. Estimación de normales
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
    )
    #radius: El radio esférico máximo de búsqueda.
    # #max_nn: El número máximo de vecinos más cercanos (nearest neighbors) a considerar dentro de ese radio.
    
    # 3. Cálculo de características FPFH
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
    )
    
    return pcd_down, pcd_fpfh

# Aplicamos el preprocesamiento tanto a la fuente (source) como al objetivo (target)
source_down_R, source_fpfh_R = preprocess_point_cloud(source_R, voxel_size_R)
target_down_R, target_fpfh_R = preprocess_point_cloud(target_R, voxel_size_R)

draw_registrations(source_down_R, target_down_R)

In [49]:
####
# Call RANSAC here
#####

distance_threshold = voxel_size_R * 1.5 #distancia euclidiana máxima permitida entre dos puntos para que el algoritmo los considere coincidentes (inliners)

point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
#point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()

ransac_result_R = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down_R, target_down_R,
    source_fpfh_R, target_fpfh_R,
    True,
    distance_threshold,
    point_to_point,
    #ransac_n=3, #RANSAC toma al azar n=3 pares de correspondencias sugeridas por los descriptores FPFH. 
    )

draw_registrations(source_R, target_R, ransac_result_R.transformation, True)

Car 1 and Car 2

In [50]:
source_car = o3d.io.read_point_cloud("global_registration/car1.ply")
target_car = o3d.io.read_point_cloud("global_registration/car2.ply")

# Used for downsampling.
voxel_size_car = 0.1 #change this value to a bigger number to have less points and get fastest resoults

# Show models side by side
draw_registrations(source_car, target_car)

In [51]:
####
# Downsample and find features here
####
# Función auxiliar para preprocesar la nube de puntos (downsampling, normales y FPFH)
def preprocess_point_cloud(pcd, voxel_size):
    # 1. Reducción de muestreo (Voxel downsampling)
    pcd_down = pcd.voxel_down_sample(voxel_size)
    
    # 2. Estimación de normales
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
    )
    #radius: El radio esférico máximo de búsqueda.
    # #max_nn: El número máximo de vecinos más cercanos (nearest neighbors) a considerar dentro de ese radio.
    
    # 3. Cálculo de características FPFH
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
    )
    
    return pcd_down, pcd_fpfh

# Aplicamos el preprocesamiento tanto a la fuente (source) como al objetivo (target)
source_down_car, source_fpfh_car = preprocess_point_cloud(source_car, voxel_size_car)
target_down_car, target_fpfh_car = preprocess_point_cloud(target_car, voxel_size_car)

draw_registrations(source_down_car, target_down_car)

In [ ]:
####
# Call RANSAC here
#####

distance_threshold_car = voxel_size_car * 1.5 #distancia euclidiana máxima permitida entre dos puntos para que el algoritmo los considere coincidentes (inliners)

point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()

ransac_result_car = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down_car, target_down_car,
    source_fpfh_car, target_fpfh_car,
    True,
    distance_threshold_car,
    point_to_point,
    #ransac_n=3, #RANSAC toma al azar n=3 pares de correspondencias sugeridas por los descriptores FPFH. 
    )

draw_registrations(source_car, target_car, ransac_result_car.transformation, True) #It is drawn the original point clouds, not the downsampled ones, to see the final result of the registration.

[Open3D WARNING] GLFW Error: WGL: Failed to make context current: Controlador no válido. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: La operación de transformación solicitada no es compatible. 
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: Controlador no válido. 


# Task 2

Local Registration of both images

In [53]:
# Umbral de distancia más estricto para el registro local
icp_distance_threshold = voxel_size_R * 0.4

# Se utiliza ransac_result.transformation como la matriz "init" para arrancar ICP
icp_result_R = o3d.pipelines.registration.registration_icp(
    source_down_R, target_down_R, 
    icp_distance_threshold,
    ransac_result_R.transformation, #instead of np.identity(4), we use the transformation from RANSAC as the initial guess for ICP
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

print("Transformación Refinada (ICP Local):")
print(icp_result_R.transformation)

# Visualizar el resultado final tras ambas alineaciones
draw_registrations(source_R, target_R, icp_result_R.transformation, True)

Transformación Refinada (ICP Local):
[[-0.54187265  0.84043805  0.00615708  0.64722043]
 [-0.21624568 -0.14649618  0.9652858   0.80610217]
 [ 0.8121649   0.52173054  0.26112337 -1.48338314]
 [ 0.          0.          0.          1.        ]]
[Open3D WARNING] GLFW Error: WGL: Failed to make context current: La operación de transformación solicitada no es compatible. 


In [55]:
# Umbral de distancia más estricto para el registro local
icp_distance_threshold = voxel_size_car * 0.4

# Se utiliza ransac_result.transformation como la matriz "init" para arrancar ICP
icp_result_car = o3d.pipelines.registration.registration_icp(
    source_down_car, target_down_car, 
    icp_distance_threshold,
    ransac_result_car.transformation, #instead of np.identity(4), we use the transformation from RANSAC as the initial guess for ICP
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

print("Transformación Refinada (ICP Local):")
print(icp_result_car.transformation)

# Visualizar el resultado final tras ambas alineaciones
draw_registrations(source_car, target_car, icp_result_car.transformation, True)

Transformación Refinada (ICP Local):
[[ 1.86491575e-03  5.59878408e-03  9.99982588e-01  2.00194641e+00]
 [ 9.99987833e-01  4.55625166e-03 -1.89043544e-03  1.99333912e+00]
 [-4.56675646e-03  9.99973947e-01 -5.59021894e-03  2.20386897e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
